# acai.storage — Test & Examples Notebook

Interactive test suite for the hexagonal storage module.
Run each cell top-to-bottom. A temp directory is used for all file I/O and cleaned up at the end.

**Architecture under test:**

| Layer | Component | Description |
|-------|-----------|-------------|
| Port | `StoragePort` | Abstract read/write/exists interface |
| Config | `StorageConfig` | Encoding, backup, size limit, extensions |
| Adapter | `LocalFileStorage` | Atomic writes, backups, JSON, validation |
| Adapter (stub) | `S3Storage` | Placeholder for future S3 support |
| Factory | `create_storage()` | Composition root |

## 0 — Setup & Imports

In [ ]:
import sys, shutil, json, tempfile
from pathlib import Path
from dataclasses import dataclass

# Ensure acai is importable
_project_root = Path.cwd()
while _project_root.name != "acai" and _project_root != _project_root.parent:
    _project_root = _project_root.parent
_shared_python = _project_root.parent  # …/shared/python
if str(_shared_python) not in sys.path:
    sys.path.insert(0, str(_shared_python))

from acai.logging import create_logger, LoggerConfig, LogLevel
from acai.storage import (
    create_storage,
    StoragePort,
    StorageConfig,
    StorageError,
    FileOperationError,
    ValidationError,
)
from acai.storage.adapters.outbound.local_file_storage import LocalFileStorage

# Shared logger for all tests
_logger = create_logger(LoggerConfig(service_name="storage-test", log_level=LogLevel.DEBUG))

# Shared temp directory — cleaned up in the last cell
WORK_DIR = Path(tempfile.mkdtemp(prefix="acai_storage_test_"))
print(f"Working directory: {WORK_DIR}")

# Test result tracker
_results: list[tuple[str, bool, str]] = []

def _record(name: str, passed: bool, detail: str = "") -> None:
    _results.append((name, passed, detail))
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status}  {name}" + (f"  — {detail}" if detail else ""))

print("Imports OK ✔")

Working directory: C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8
Imports OK ✔


## 1 — Factory: `create_storage()`

Create a `StoragePort` via the factory with default and custom configs.

In [ ]:
# Default factory → LocalFileStorage
storage = create_storage(_logger)
_record("Factory — default returns LocalFileStorage", isinstance(storage, LocalFileStorage))

# Custom config
cfg = StorageConfig(
    encoding="utf-8",
    backup_enabled=True,
    max_file_size=1 * 1024 * 1024,  # 1 MB
    allowed_extensions={"txt", "json", "md"},
)
storage = create_storage(_logger, cfg)
_record("Factory — custom config accepted", isinstance(storage, LocalFileStorage))

✅ PASS  Factory — default returns LocalFileStorage
✅ PASS  Factory — custom config accepted


## 2 — Save & Read Text Files

Write a plain text file, read it back, and verify round-trip integrity.

In [ ]:
txt_path = WORK_DIR / "hello.txt"
original = "Grüezi, Welt!\nThis is a test file with UTF-8: äöü"

storage.save(txt_path, original)
read_back = storage.read(txt_path)

_record("Text save/read — round-trip matches", read_back == original)
_record("Text save/read — file exists", storage.exists(txt_path))
print(f"Content:\n{read_back}")

2026-03-23 15:09:38,908 - DEBUG - Saving file | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\hello.txt
2026-03-23 15:09:38,909 - INFO - File saved successfully | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\hello.txt
2026-03-23 15:09:38,911 - DEBUG - Reading file | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\hello.txt
2026-03-23 15:09:38,911 - INFO - File read successfully | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\hello.txt


✅ PASS  Text save/read — round-trip matches
✅ PASS  Text save/read — file exists
Content:
Grüezi, Welt!
This is a test file with UTF-8: äöü


## 3 — Save & Read JSON (dict and dataclasses)

Persist a dict and a list of dataclasses, then deserialise back into typed objects.

In [ ]:
# ── 3a: plain dict ────────────────────────────────────────────────
json_path = WORK_DIR / "config.json"
data = {"project": "law-bot", "version": 2, "languages": ["de", "fr", "it"]}

storage.save_json(json_path, data)
loaded = storage.read_json(json_path)

_record("JSON dict — round-trip", loaded == data, f"loaded={loaded}")

# ── 3b: list of dataclasses ──────────────────────────────────────
@dataclass
class LawRef:
    sr_number: str
    title: str
    language: str

refs = [
    LawRef("SR-210", "Zivilgesetzbuch", "de"),
    LawRef("SR-311", "Strafgesetzbuch", "de"),
]

refs_path = WORK_DIR / "refs.json"
storage.save_json(refs_path, refs)

# Read back as typed objects
loaded_refs = storage.read_json(refs_path, data_type=LawRef)

_record("JSON dataclass — count matches", len(loaded_refs) == 2)
_record(
    "JSON dataclass — first item correct",
    loaded_refs[0].sr_number == "SR-210" and loaded_refs[0].title == "Zivilgesetzbuch",
    f"first={loaded_refs[0]}",
)
print(f"Loaded refs: {loaded_refs}")

2026-03-23 15:09:38,919 - INFO - JSON saved successfully | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\config.json
2026-03-23 15:09:38,920 - DEBUG - Reading file | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\config.json
2026-03-23 15:09:38,921 - INFO - File read successfully | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\config.json
2026-03-23 15:09:38,923 - INFO - JSON saved successfully | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\refs.json
2026-03-23 15:09:38,924 - DEBUG - Reading file | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\refs.json
2026-03-23 15:09:38,925 - INFO - File read successfully | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\refs.json


✅ PASS  JSON dict — round-trip  — loaded={'project': 'law-bot', 'version': 2, 'languages': ['de', 'fr', 'it']}
✅ PASS  JSON dataclass — count matches
✅ PASS  JSON dataclass — first item correct  — first=LawRef(sr_number='SR-210', title='Zivilgesetzbuch', language='de')
Loaded refs: [LawRef(sr_number='SR-210', title='Zivilgesetzbuch', language='de'), LawRef(sr_number='SR-311', title='Strafgesetzbuch', language='de')]


## 4 — Non-Existent Files Return Safe Defaults

`read()` returns `""` and `read_json()` returns `{}` for missing files — no exceptions.

In [ ]:
missing_txt = storage.read(WORK_DIR / "nope.txt")
missing_json = storage.read_json(WORK_DIR / "nope.json")

_record("Missing text — returns empty string", missing_txt == "")
_record("Missing JSON — returns empty dict", missing_json == {})
_record("Missing file — exists() is False", not storage.exists(WORK_DIR / "nope.txt"))

2026-03-23 15:09:38,930 - INFO - File does not exist, returning empty string | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\nope.txt
2026-03-23 15:09:38,931 - INFO - File does not exist, returning empty dict | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\nope.json


✅ PASS  Missing text — returns empty string
✅ PASS  Missing JSON — returns empty dict
✅ PASS  Missing file — exists() is False


## 5 — Nested Directories (auto-created)

`save()` creates parent directories automatically — no need to `mkdir` in advance.

In [ ]:
nested_path = WORK_DIR / "deep" / "nested" / "dir" / "file.txt"
storage.save(nested_path, "created through nested dirs")

_record("Nested dirs — file created", storage.exists(nested_path))
_record("Nested dirs — content correct", storage.read(nested_path) == "created through nested dirs")

2026-03-23 15:09:38,938 - DEBUG - Saving file | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\deep\nested\dir\file.txt
2026-03-23 15:09:38,940 - INFO - File saved successfully | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\deep\nested\dir\file.txt
2026-03-23 15:09:38,941 - DEBUG - Reading file | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\deep\nested\dir\file.txt
2026-03-23 15:09:38,942 - INFO - File read successfully | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\deep\nested\dir\file.txt


✅ PASS  Nested dirs — file created
✅ PASS  Nested dirs — content correct


## 6 — Extension Validation

When `allowed_extensions` is configured, saving a file with a forbidden extension raises `ValidationError`.

In [ ]:
restricted_cfg = StorageConfig(allowed_extensions={"txt", "json"})
restricted = LocalFileStorage(logger=_logger, config=restricted_cfg)

# Allowed extension → should work
restricted.save(WORK_DIR / "ok.txt", "allowed")
_record("Ext validation — .txt allowed", storage.exists(WORK_DIR / "ok.txt"))

# Forbidden extension → should raise
caught = False
try:
    restricted.save(WORK_DIR / "bad.exe", "blocked")
except ValidationError as exc:
    caught = True
    print(f"  Caught: {exc}")

_record("Ext validation — .exe blocked", caught)

2026-03-23 15:09:38,947 - DEBUG - Saving file | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\ok.txt
2026-03-23 15:09:38,949 - INFO - File saved successfully | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\ok.txt


✅ PASS  Ext validation — .txt allowed
  Caught: Extension '.exe' not allowed. Allowed: {'json', 'txt'}
✅ PASS  Ext validation — .exe blocked


## 7 — File Size Limit (atomic write)

Writes exceeding `max_file_size` are rejected **before** replacing the original — the atomic write rolls back.

In [ ]:
tiny_cfg = StorageConfig(max_file_size=50)  # 50 bytes
tiny_store = LocalFileStorage(logger=_logger, config=tiny_cfg)

caught_size = False
try:
    tiny_store.save(WORK_DIR / "big.txt", "x" * 100)  # 100 bytes > 50
except FileOperationError as exc:
    caught_size = True
    print(f"  Caught: {exc}")

_record("Size limit — oversized write rejected", caught_size)
_record("Size limit — original untouched", not storage.exists(WORK_DIR / "big.txt"))

2026-03-23 15:09:38,955 - DEBUG - Saving file | path=C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8\big.txt


  Caught: File size exceeds limit of 50 bytes
✅ PASS  Size limit — oversized write rejected
✅ PASS  Size limit — original untouched


## 8 — Context Manager Cleanup

In [ ]:
import tempfile, shutil

ctx_dir = Path(tempfile.mkdtemp())
with LocalFileStorage(logger=_logger) as ctx_store:
    ctx_store.save(ctx_dir / "managed.txt", "inside context manager")
    value = ctx_store.read(ctx_dir / "managed.txt")
    _record("Context manager — read inside block", value == "inside context manager")

# After exiting, the adapter is still usable (it's not a temp-dir manager itself)
_record("Context manager — exit succeeded", True)
shutil.rmtree(ctx_dir, ignore_errors=True)

2026-03-23 15:09:38,963 - DEBUG - Saving file | path=C:\Users\micha\AppData\Local\Temp\tmp5dgr_7np\managed.txt
2026-03-23 15:09:38,965 - INFO - File saved successfully | path=C:\Users\micha\AppData\Local\Temp\tmp5dgr_7np\managed.txt
2026-03-23 15:09:38,966 - DEBUG - Reading file | path=C:\Users\micha\AppData\Local\Temp\tmp5dgr_7np\managed.txt
2026-03-23 15:09:38,967 - INFO - File read successfully | path=C:\Users\micha\AppData\Local\Temp\tmp5dgr_7np\managed.txt


✅ PASS  Context manager — read inside block
✅ PASS  Context manager — exit succeeded


## 9 — S3 Storage Stub (NotImplementedError)

In [ ]:
from acai.storage.adapters.outbound.s3_storage import S3Storage

s3 = S3Storage(logger=_logger, bucket="my-bucket", prefix="data/")

caught_save = False
try:
    s3.save(Path("test.txt"), "hello")
except NotImplementedError:
    caught_save = True

caught_read = False
try:
    s3.read(Path("test.txt"))
except NotImplementedError:
    caught_read = True

caught_exists = False
try:
    s3.exists(Path("test.txt"))
except NotImplementedError:
    caught_exists = True

_record("S3 stub — save raises NotImplementedError", caught_save)
_record("S3 stub — read raises NotImplementedError", caught_read)
_record("S3 stub — exists raises NotImplementedError", caught_exists)

✅ PASS  S3 stub — save raises NotImplementedError
✅ PASS  S3 stub — read raises NotImplementedError
✅ PASS  S3 stub — exists raises NotImplementedError


## Cleanup

In [ ]:
import shutil

shutil.rmtree(WORK_DIR, ignore_errors=True)
print(f"Removed temp directory: {WORK_DIR}")
print(f"Directory exists after cleanup: {WORK_DIR.exists()}")

Removed temp directory: C:\Users\micha\AppData\Local\Temp\acai_storage_test_lhufjvj8
Directory exists after cleanup: False


## Summary

In [ ]:
passed = sum(1 for _, ok, _ in _results if ok)
failed = sum(1 for _, ok, _ in _results if not ok)
total  = len(_results)

print(f"\n{'='*60}")
print(f"  RESULTS: {passed}/{total} passed, {failed} failed")
print(f"{'='*60}\n")

for name, ok, detail in _results:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {name}" + (f"  — {detail}" if detail else ""))


  RESULTS: 21/21 passed, 0 failed

  [PASS] Factory — default returns LocalFileStorage
  [PASS] Factory — custom config accepted
  [PASS] Text save/read — round-trip matches
  [PASS] Text save/read — file exists
  [PASS] JSON dict — round-trip  — loaded={'project': 'law-bot', 'version': 2, 'languages': ['de', 'fr', 'it']}
  [PASS] JSON dataclass — count matches
  [PASS] JSON dataclass — first item correct  — first=LawRef(sr_number='SR-210', title='Zivilgesetzbuch', language='de')
  [PASS] Missing text — returns empty string
  [PASS] Missing JSON — returns empty dict
  [PASS] Missing file — exists() is False
  [PASS] Nested dirs — file created
  [PASS] Nested dirs — content correct
  [PASS] Ext validation — .txt allowed
  [PASS] Ext validation — .exe blocked
  [PASS] Size limit — oversized write rejected
  [PASS] Size limit — original untouched
  [PASS] Context manager — read inside block
  [PASS] Context manager — exit succeeded
  [PASS] S3 stub — save raises NotImplementedError
  [PA